# News Modeling

Topic modeling involves **extracting features from document terms** and using
mathematical structures and frameworks like matrix factorization and SVD to generate **clusters or groups of terms** that are distinguishable from each other and these clusters of words form topics or concepts

Topic modeling is a method for **unsupervised classification** of documents, similar to clustering on numeric data

These concepts can be used to interpret the main **themes** of a corpus and also make **semantic connections among words that co-occur together** frequently in various documents

Topic modeling can help in the following areas:
- discovering the **hidden themes** in the collection
- **classifying** the documents into the discovered themes
- using the classification to **organize/summarize/search** the documents

Frameworks and algorithms to build topic models:
- Latent semantic indexing
- Latent Dirichlet allocation
- Non-negative matrix factorization

## Latent Dirichlet Allocation (LDA)
The latent Dirichlet allocation (LDA) technique is a **generative probabilistic model** where each **document is assumed to have a combination of topics** similar to a probabilistic latent semantic indexing model

In simple words, the idea behind LDA is that of two folds:
- each **document** can be described by a **distribution of topics**
- each **topic** can be described by a **distribution of words**

### LDA Algorithm

- 1. For each document, **randomly initialize each word to one of the K topics** (k is chosen beforehand)
- 2. For each document D, go through each word w and compute:
    - **P(T |D)** , which is a proportion of words in D assigned to topic T
    - **P(W |T )** , which is a proportion of assignments to topic T over all documents having the word W
- **Reassign word W with topic T** with probability P(T |D)´ P(W |T ) considering all other words and their topic assignments

![LDA](https://raw.githubusercontent.com/subashgandyer/datasets/main/images/LDA.png)

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Stop words removal
    - Email removal
    - Non-alphabetic words removal
    - Tokenize
    - Lowercase
    - BiGrams & TriGrams
    - Lemmatization
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Evaluate the Topic Model
    - Model Perplexity
    - Topic Coherence
- Visualize the topics

### Install the necessary library

In [1]:
! pip install pyLDAvis gensim spacy

### Import the libraries

In [31]:
from pprint import pprint
import re
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer

import gensim
from gensim import corpora, models
from gensim.corpora import Dictionary

from gensim.models import LdaModel, LsiModel
from gensim.models import CoherenceModel

from gensim.models.phrases import Phrases, Phraser

import pyLDAvis
import pyLDAvis.gensim

import spacy

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /Users/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Download the dataset
Dataset: https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json

#### 20-Newsgroups dataset
- 11K newsgroups posts
- 20 news topics

In [32]:
df = pd.read_json('newsgroups.json')
df.head()

,content,target,target_names
0,From: lerxst@wam.umd.edu (where's my thing)\nS...,7,rec.autos
1,From: guykuo@carson.u.washington.edu (Guy Kuo)...,4,comp.sys.mac.hardware
2,From: twillis@ec.ecn.purdue.edu (Thomas E Will...,4,comp.sys.mac.hardware
3,From: jgreen@amber (Joe Green)\nSubject: Re: W...,1,comp.graphics
4,From: jcm@head-cfa.harvard.edu (Jonathan McDow...,14,sci.space


### Load the dataset

In [33]:
# dataset is in the samefolder as my notebook
# instead of loading, I explored the dataset
print(df.shape)

print(df.columns)

print(df.dtypes)

print(df.isnull().sum())

print(df['content'].iloc[0])

(11314, 3)
Index(['content', 'target', 'target_names'], dtype='object')
content         object
target           int64
target_names    object
dtype: object
content         0
target          0
target_names    0
dtype: int64
From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----







### Preprocess the data

### Email Removal

In [34]:
import re

def remove_emails(text):
    return re.sub(r'\S+@\S+', '', text)

df['content'] = df['content'].apply(remove_emails)

print(df['content'].iloc[0])

From:  (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----







### Newline Removal

In [35]:
def remove_newlines(text):
    return re.sub(r'\n', ' ', text)

df['content'] = df['content'].apply(remove_newlines)

print(df['content'].iloc[0])

From:  (where's my thing) Subject: WHAT car is this!? Nntp-Posting-Host: rac3.wam.umd.edu Organization: University of Maryland, College Park Lines: 15   I was wondering if anyone out there could enlighten me on this car I saw the other day. It was a 2-door sports car, looked to be from the late 60s/ early 70s. It was called a Bricklin. The doors were really small. In addition, the front bumper was separate from the rest of the body. This is  all I know. If anyone can tellme a model name, engine specs, years of production, where this car is made, history, or whatever info you have on this funky looking car, please e-mail.  Thanks, - IL    ---- brought to you by your neighborhood Lerxst ----     


### Single Quotes Removal

In [36]:
def remove_single_quotes(text):
    return re.sub(r"\'", '', text)

df['content'] = df['content'].apply(remove_single_quotes)

print(df['content'].iloc[0])


From:  (wheres my thing) Subject: WHAT car is this!? Nntp-Posting-Host: rac3.wam.umd.edu Organization: University of Maryland, College Park Lines: 15   I was wondering if anyone out there could enlighten me on this car I saw the other day. It was a 2-door sports car, looked to be from the late 60s/ early 70s. It was called a Bricklin. The doors were really small. In addition, the front bumper was separate from the rest of the body. This is  all I know. If anyone can tellme a model name, engine specs, years of production, where this car is made, history, or whatever info you have on this funky looking car, please e-mail.  Thanks, - IL    ---- brought to you by your neighborhood Lerxst ----     


### Tokenize
- Create **sent_to_words()** 
    - Use **gensim.utils.simple_preprocess**
    - Use **generator** instead of an usual function

In [37]:
def sent_to_words(sentences):
    for sentence in sentences:
        yield gensim.utils.simple_preprocess(str(sentence), deacc=True)

data = df['content'].values.tolist()
data_words = list(sent_to_words(data))

print(data_words[0])


['from', 'wheres', 'my', 'thing', 'subject', 'what', 'car', 'is', 'this', 'nntp', 'posting', 'host', 'rac', 'wam', 'umd', 'edu', 'organization', 'university', 'of', 'maryland', 'college', 'park', 'lines', 'was', 'wondering', 'if', 'anyone', 'out', 'there', 'could', 'enlighten', 'me', 'on', 'this', 'car', 'saw', 'the', 'other', 'day', 'it', 'was', 'door', 'sports', 'car', 'looked', 'to', 'be', 'from', 'the', 'late', 'early', 'it', 'was', 'called', 'bricklin', 'the', 'doors', 'were', 'really', 'small', 'in', 'addition', 'the', 'front', 'bumper', 'was', 'separate', 'from', 'the', 'rest', 'of', 'the', 'body', 'this', 'is', 'all', 'know', 'if', 'anyone', 'can', 'tellme', 'model', 'name', 'engine', 'specs', 'years', 'of', 'production', 'where', 'this', 'car', 'is', 'made', 'history', 'or', 'whatever', 'info', 'you', 'have', 'on', 'this', 'funky', 'looking', 'car', 'please', 'mail', 'thanks', 'il', 'brought', 'to', 'you', 'by', 'your', 'neighborhood', 'lerxst']


### Stop words Removal
- Extend the stop words corpus with the following words
    - from
    - subject
    - re
    - edu
    - use

In [38]:
stop_words = stopwords.words('english')

stop_words.extend(['from', 'subject', 're', 'edu', 'use'])

def remove_stopwords(texts):
    return [[word for word in doc if word not in stop_words] 
            for doc in texts]

data_words_nostops = remove_stopwords(data_words)

print(data_words_nostops[0])

['wheres', 'thing', 'car', 'nntp', 'posting', 'host', 'rac', 'wam', 'umd', 'organization', 'university', 'maryland', 'college', 'park', 'lines', 'wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sports', 'car', 'looked', 'late', 'early', 'called', 'bricklin', 'doors', 'really', 'small', 'addition', 'front', 'bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'specs', 'years', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please', 'mail', 'thanks', 'il', 'brought', 'neighborhood', 'lerxst']


#### remove_stopwords( )

In [39]:
def remove_stopwords(texts):
    return [[word for word in doc if word not in stop_words] 
            for doc in texts]
    return None   # This line NEVER runs!


In [40]:
data_words_nostops = remove_stopwords(data_words)

print("Before:", data_words[0])

print("After:", data_words_nostops[0])

Before: ['from', 'wheres', 'my', 'thing', 'subject', 'what', 'car', 'is', 'this', 'nntp', 'posting', 'host', 'rac', 'wam', 'umd', 'edu', 'organization', 'university', 'of', 'maryland', 'college', 'park', 'lines', 'was', 'wondering', 'if', 'anyone', 'out', 'there', 'could', 'enlighten', 'me', 'on', 'this', 'car', 'saw', 'the', 'other', 'day', 'it', 'was', 'door', 'sports', 'car', 'looked', 'to', 'be', 'from', 'the', 'late', 'early', 'it', 'was', 'called', 'bricklin', 'the', 'doors', 'were', 'really', 'small', 'in', 'addition', 'the', 'front', 'bumper', 'was', 'separate', 'from', 'the', 'rest', 'of', 'the', 'body', 'this', 'is', 'all', 'know', 'if', 'anyone', 'can', 'tellme', 'model', 'name', 'engine', 'specs', 'years', 'of', 'production', 'where', 'this', 'car', 'is', 'made', 'history', 'or', 'whatever', 'info', 'you', 'have', 'on', 'this', 'funky', 'looking', 'car', 'please', 'mail', 'thanks', 'il', 'brought', 'to', 'you', 'by', 'your', 'neighborhood', 'lerxst']
After: ['wheres', 'thin

### Bigrams
- Use **gensim.models.Phrases**
- 100 as threshold

In [41]:
bigram = Phrases(data_words_nostops, threshold=100)

bigram_mod = Phraser(bigram)

def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

data_words_bigrams = make_bigrams(data_words_nostops)

print(data_words_bigrams[0])

['wheres', 'thing', 'car', 'nntp_posting', 'host', 'rac_wam', 'umd', 'organization', 'university', 'maryland_college', 'park', 'lines', 'wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sports', 'car', 'looked', 'late', 'early', 'called', 'bricklin', 'doors', 'really', 'small', 'addition', 'front_bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'specs', 'years', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please', 'mail', 'thanks', 'il', 'brought', 'neighborhood', 'lerxst']


#### make_bigrams( )

In [42]:
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]
    return None

In [43]:
data_words_bigrams = make_bigrams(data_words_nostops)
print("Before:", data_words_nostops[0])
print("After: ", data_words_bigrams[0])

Before: ['wheres', 'thing', 'car', 'nntp', 'posting', 'host', 'rac', 'wam', 'umd', 'organization', 'university', 'maryland', 'college', 'park', 'lines', 'wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sports', 'car', 'looked', 'late', 'early', 'called', 'bricklin', 'doors', 'really', 'small', 'addition', 'front', 'bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'specs', 'years', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please', 'mail', 'thanks', 'il', 'brought', 'neighborhood', 'lerxst']
After:  ['wheres', 'thing', 'car', 'nntp_posting', 'host', 'rac_wam', 'umd', 'organization', 'university', 'maryland_college', 'park', 'lines', 'wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sports', 'car', 'looked', 'late', 'early', 'called', 'bricklin', 'doors', 'really', 'small', 'addition', 'front_bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'te

### Lemmatization
- Use spacy
    - Download spacy en model (if you have not done that before)
    - Load the spacy model

In [14]:
#! python -m spacy download en
!python -m spacy download en_core_web_sm

/opt/anaconda3/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=90653) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 7.7 MB/s  0:00:01m 7.7 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [44]:
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

#### lemmatizaton( )

In [45]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [46]:
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [47]:
print(data_lemmatized[:1])

[['s', 'thing', 'car', 'nntp_poste', 'host', 'park', 'line', 'wonder', 'enlighten', 'car', 'see', 'day', 'door', 'sport', 'car', 'look', 'late', 'early', 'call', 'bricklin', 'door', 'really', 'small', 'addition', 'separate', 'rest', 'body', 'know', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'make', 'history', 'info', 'funky', 'look', 'car', 'mail', 'thank', 'bring', 'neighborhood', 'lerxst']]


### Create a Dictionary

In [19]:
id2word = corpora.Dictionary(data_lemmatized)

print(id2word)

print("Number of unique words:", len(id2word))

print(id2word[0])   # word at ID 0
print(id2word[1])   # word at ID 1

Dictionary<50517 unique tokens: ['addition', 'body', 'bricklin', 'bring', 'call']...>
Number of unique words: 50517
addition
body


### Filter low-frequency words

In [28]:
id2word.filter_extremes(no_below=10, no_above=0.5)

print(id2word)

Dictionary<8178 unique tokens: ['addition', 'body', 'bring', 'call', 'car']...>


### Create Corpus

In [29]:
corpus = [id2word.doc2bow(text) for text in data_lemmatized]

print(corpus[:1])


[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 5), (5, 1), (6, 2), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 2), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1)]]


### Create Index 2 word dictionary

In [53]:
temp = id2word[0]

id2word_index = id2word.id2token

print(id2word_index)

{0: 'addition', 1: 'body', 2: 'bring', 3: 'call', 4: 'car', 5: 'day', 6: 'door', 7: 'early', 8: 'engine', 9: 'enlighten', 10: 'history', 11: 'host', 12: 'info', 13: 'know', 14: 'late', 15: 'look', 16: 'mail', 17: 'make', 18: 'model', 19: 'name', 20: 'neighborhood', 21: 'nntp_poste', 22: 'park', 23: 'production', 24: 'really', 25: 'rest', 26: 's', 27: 'see', 28: 'separate', 29: 'small', 30: 'spec', 31: 'sport', 32: 'thank', 33: 'thing', 34: 'wonder', 35: 'year', 36: 'acceleration', 37: 'adapter', 38: 'add', 39: 'answer', 40: 'attain', 41: 'base', 42: 'brave', 43: 'brief', 44: 'card', 45: 'clock', 46: 'cpu', 47: 'detail', 48: 'do', 49: 'especially', 50: 'experience', 51: 'fair', 52: 'final', 53: 'floppy', 54: 'floppy_disk', 55: 'functionality', 56: 'heat_sink', 57: 'hour', 58: 'keyword', 59: 'knowledge', 60: 'message', 61: 'network', 62: 'next', 63: 'number', 64: 'oscillator', 65: 'poll', 66: 'procedure', 67: 'rat', 68: 'report', 69: 'request', 70: 'send', 71: 'share', 72: 'si', 73: 'sou

### Build a News Topic Model

#### LdaModel
- **num_topics** : this is the number of topics you need to define beforehand
- **chunksize** : the number of documents to be used in each training chunk
- **alpha** : this is the hyperparameters that affect the sparsity of the topics
- **passess** : total number of training assess

In [23]:
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=20,
    chunksize=100,
    alpha='auto',
    passes=10
)

### Print the Keyword in the 10 topics

In [24]:
pprint(lda_model.print_topics())

[(0,
  '0.033*"need" + 0.032*"use" + 0.026*"get" + 0.024*"new" + 0.022*"system" + '
  '0.020*"also" + 0.019*"work" + 0.016*"help" + 0.015*"look" + 0.013*"high"'),
 (1,
  '0.086*"player" + 0.051*"black" + 0.048*"publish" + 0.039*"tape" + '
  '0.039*"record" + 0.037*"convince" + 0.037*"department" + 0.036*"dept" + '
  '0.032*"library" + 0.032*"medium"'),
 (2,
  '0.024*"know" + 0.024*"say" + 0.022*"article" + 0.022*"think" + 0.022*"go" + '
  '0.018*"see" + 0.017*"get" + 0.017*"make" + 0.016*"time" + 0.016*"people"'),
 (3,
  '0.054*"money" + 0.049*"pay" + 0.035*"bike" + 0.028*"israeli" + 0.025*"area" '
  '+ 0.025*"drug" + 0.024*"national" + 0.023*"city" + 0.021*"society" + '
  '0.021*"country"'),
 (4,
  '0.126*"book" + 0.050*"graphic" + 0.047*"school" + 0.039*"picture" + '
  '0.038*"scan" + 0.038*"format" + 0.036*"author" + 0.027*"printer" + '
  '0.027*"dealer" + 0.026*"air"'),
 (5,
  '0.718*"ax" + 0.037*"entry" + 0.021*"brain" + 0.014*"trust" + '
  '0.013*"upgrade" + 0.012*"load" + 0.011*

## Evaluation of Topic Models
- Model Perplexity
- Topic Coherence

### Model Perplexity

Model perplexity is a measurement of **how well** a **probability distribution** or probability model **predicts a sample**

In [25]:
perplexity = lda_model.log_perplexity(corpus)
print(f"Model Perplexity: {perplexity}")


Model Perplexity: -11.458704550359894


### Topic Coherence
Topic Coherence measures score a single topic by measuring the **degree of semantic similarity** between **high scoring words** in the topic.

In [26]:
coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=data_lemmatized,
    dictionary=id2word,
    coherence='c_v'
)

coherence_lda = coherence_model_lda.get_coherence()
print(f"Topic Coherence Score: {coherence_lda}")


Topic Coherence Score: 0.4677369262044645


### Visualize the Topic Model
- Use **pyLDAvis**
    - designed to help users **interpret the topics** in a topic model that has been fit to a corpus of text data
    - extracts information from a fitted LDA topic model to inform an interactive web-based visualization

In [27]:
import warnings
warnings.filterwarnings("ignore")

pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)

vis


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.290892 -0.029717       1        1  28.973443
0     -0.251053  0.091453       2        1  12.141467
17    -0.209917 -0.106785       3        1  10.634841
14    -0.107015 -0.016954       4        1   5.113453
12    -0.129180  0.038351       5        1   5.016867
19    -0.073340  0.055646       6        1   4.878224
5      0.143616  0.013672       7        1   4.865235
10    -0.074829  0.220433       8        1   4.331874
6     -0.038571 -0.204869       9        1   3.637457
11     0.026173  0.006583      10        1   3.212383
15     0.047972  0.249127      11        1   2.889729
7      0.109675 -0.121801      12        1   2.515835
3      0.067013 -0.161283      13        1   2.080585
16     0.022118 -0.245587      14        1   1.850485
13     0.121063  0.108098      15        1   1.659649
9      0.101184  0.117077      16        1   1.410551
4      0.152997  0.002692      17        1   1.407641
8      0.101885 -0.000550      18        1   1.291390
1      0.152783 -0.002613      19        1   1.194324
18     0.128316 -0.012974      20        1   0.894567, topic_info=             Term          Freq         Total Category  logprob  loglift
3174           ax  38540.000000  38540.000000  Default  30.0000  30.0000
21     nntp_poste   5943.000000   5943.000000  Default  29.0000  29.0000
11           host   5157.000000   5157.000000  Default  28.0000  28.0000
228       believe   3288.000000   3288.000000  Default  27.0000  27.0000
307          file   2489.000000   2489.000000  Default  26.0000  26.0000
...           ...           ...           ...      ...      ...      ...
2185  convenience     76.100903     77.455116  Topic20  -4.8648   4.6989
1013   population    202.736333    340.359900  Topic20  -3.8850   4.1985
63         number    251.154047   2913.448042  Topic20  -3.6708   2.2656
696          high    174.421077   1869.420472  Topic20  -4.0354   2.3447
1237         thus    106.602761    610.760105  Topic20  -4.5278   2.9710

[886 rows x 6 columns], token_table=      Topic      Freq       Term
term                            
444      18  0.988657   absolute
80       10  0.108948     access
80       11  0.889996     access
798      13  0.991881   accident
799       3  0.996621    account
...     ...       ...        ...
35       10  0.026302       year
35       13  0.008147       year
35       14  0.030259       year
677       5  0.991387  yesterday
1768      7  0.998350       zone

[1201 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 1, 18, 15, 13, 20, 6, 11, 7, 12, 16, 8, 4, 17, 14, 10, 5, 9, 2, 19])